# 01 — Training privacyFinding the optimal basis standard deviation `sigma_b*` at a fixed privacy budget(Algorithm 2 of the paper), and checking what the SNR metric predicted.The proprietary NIST high-speed camera data is replaced by MNIST, downloadedautomatically on first run. See `data/README.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import torch

from dphd import *
from dphd.data import load_dataset, split_to_tensors
from dphd.plotting import plot_sigma_sweep, plot_accuracy_vs_nmse
from dphd.privacy import snr_db

set_seed(21)
device = get_device()
print("device:", device)

## Parameters

In [ ]:
HV_DIM = 1000        # hypervector dimension D
N_EPOCHS = 4         # retraining passes T
EPSILON = 0.8        # privacy budget
DELTA = 1e-4         # privacy loss threshold
SIGMA_GRID = np.linspace(0.01, 1.2, 30)
N_SAMPLES = 5000     # subsample size before the 70/30 split
DOWNSAMPLE = 2       # 28x28 -> 14x14, i.e. 196 features

## Data

In [ ]:
X, y = load_dataset("mnist", root="../data", n_samples=N_SAMPLES, downsample=DOWNSAMPLE)
X_tr, X_te, y_tr, y_te = split_to_tensors(X, y, device=device)

n_classes = int(torch.unique(y_tr).numel())
n_features = X_tr.shape[1]
print(f"{len(X_tr)} train / {len(X_te)} test, {n_features} features, {n_classes} classes")

## Sweep sigma_bFor each candidate the pipeline is: encode -> bundle -> retrain -> add DP noiseto the class hypervectors -> measure accuracy on the held-out set.

In [ ]:
accs, snrs = [], []

for sigma_b in SIGMA_GRID:
    basis, bias = make_basis(n_features, HV_DIM, float(sigma_b), device)
    enc_tr, enc_te = encode(X_tr, basis, bias), encode(X_te, basis, bias)

    class_hvs = train_class_hvs(enc_tr, y_tr, n_classes)
    class_hvs = retrain(class_hvs, enc_tr, y_tr, n_epochs=N_EPOCHS)

    delta_f = sensitivity(enc_tr)
    mu_c = mean_class_similarity(enc_tr, y_tr)

    noisy = privatize(class_hvs, epsilon=EPSILON, delta_f=delta_f, delta=DELTA)
    acc = accuracy(enc_te, noisy, y_te)

    accs.append(acc * 100)
    snrs.append(snr_db(len(enc_tr), mu_c, HV_DIM, EPSILON, DELTA, N_EPOCHS))
    print(f"sigma_b={sigma_b:.3f}  accuracy={acc * 100:5.2f}%")

accs = np.asarray(accs)

## Result

In [ ]:
best = int(np.argmax(accs))
print(f"sigma_b* = {SIGMA_GRID[best]:.3f}  ->  {accs[best]:.2f}% at epsilon = {EPSILON}")
print(f"predicted SNR = {snrs[best]:.2f} dB")

plot_sigma_sweep(SIGMA_GRID, accs, EPSILON);

The shaded band is the range of `sigma_b` that still clears 90% accuracy. Itswidth is what the paper calls low output uncertainty: you do not have to hit`sigma_b*` exactly for the model to behave.## Privacy–accuracy trade-offSame encoder, sweeping the budget instead. Smaller epsilon means more noise andstronger privacy.

In [ ]:
SIGMA_STAR = float(SIGMA_GRID[best])

basis, bias = make_basis(n_features, HV_DIM, SIGMA_STAR, device)
enc_tr, enc_te = encode(X_tr, basis, bias), encode(X_te, basis, bias)
class_hvs = train_class_hvs(enc_tr, y_tr, n_classes)
class_hvs = retrain(class_hvs, enc_tr, y_tr, n_epochs=N_EPOCHS)
delta_f, mu_c = sensitivity(enc_tr), mean_class_similarity(enc_tr, y_tr)

print(f"{'epsilon':>8} {'SNR (dB)':>10} {'accuracy':>10}")
for eps in [0.2, 0.6, 0.8, 1.0, 2.0, 5.0, 10.0]:
    noisy = privatize(class_hvs, epsilon=eps, delta_f=delta_f, delta=DELTA)
    acc = accuracy(enc_te, noisy, y_te) * 100
    print(f"{eps:>8.1f} {snr_db(len(enc_tr), mu_c, HV_DIM, eps, DELTA, N_EPOCHS):>10.2f} {acc:>9.2f}%")